# Ultimate Combiner — Final Dataset Assembly + IMD Enrichment

Consolidates all Google Review batches, merges with the master Companies House dataset, and enriches with the **ONS Index of Multiple Deprivation (IMD 2019)** at postcode level.

**New:** Adds IMD decile, employment deprivation score, and income deprivation score per business postcode.

**Output:** `Data/Merged_Final_Dataset.xlsx`

In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import os
from pathlib import Path

if '__file__' in dir():
    REPO_ROOT = Path(__file__).resolve().parent.parent
else:
    REPO_ROOT = Path().resolve()
os.chdir(REPO_ROOT)
DATA_DIR = REPO_ROOT / "Data"
print(f"Repo root : {REPO_ROOT}")


## 1. Load & Consolidate Google Review Batches

In [ ]:
REVIEW_COLS = [' CompanyNumber','Google_Overall_Rating','Google_Price_Level',
               'Google_Review_Texts','Google_Individual_Review_Ratings']

df1      = pd.read_excel(DATA_DIR / 'Query_complete.xlsx')
df1_sub1 = pd.read_csv(DATA_DIR / 'Trial_dataset1_10000_with_GoogleReviews.csv',
                       encoding='latin-1')[REVIEW_COLS]
df1_sub2 = pd.read_csv(DATA_DIR / 'Trial_dataset10001_20000_with_GoogleReviews.csv',
                       encoding='latin-1')[REVIEW_COLS]

df1_sub = pd.concat([df1_sub1, df1_sub2]).drop_duplicates(keep=False)

df1.set_index(' CompanyNumber', inplace=True)
df1_sub.set_index(' CompanyNumber', inplace=True)
df1.update(df1_sub)
df1.reset_index(inplace=True)

df2     = pd.read_excel(DATA_DIR / 'Query_complete_03_06_26.xlsx')
df2_sub = pd.read_excel(DATA_DIR / 'Quer_file_output_03_06_26.xlsx')[REVIEW_COLS]
df2_sub = df2_sub.dropna(subset=['Google_Overall_Rating'])

final_df = pd.concat([df1, df2, df2_sub], ignore_index=True)

print(f"Consolidated review records: {len(final_df):,}")
print(f"  With rating    : {final_df['Google_Overall_Rating'].notna().sum():,}")
print(f"  Without rating : {final_df['Google_Overall_Rating'].isna().sum():,}")


## 2. Merge with Master Companies House Dataset

In [ ]:
master = pd.read_excel(DATA_DIR / 'Final Dataset.xlsx', sheet_name='Sheet1')

# Zero-pad company numbers for consistent joining
final_df[' CompanyNumber'] = final_df[' CompanyNumber'].astype(str).str.zfill(8)
master[' CompanyNumber']   = master[' CompanyNumber'].astype(str).str.zfill(8)

df_combined = final_df.merge(master, on=' CompanyNumber', how='left')
print(f"Post-merge shape: {df_combined.shape}")
print(f"Null check (top 5):")
print(df_combined.isna().sum().sort_values(ascending=False).head(5).to_string())


## 3. IMD 2019 Enrichment

Downloads the ONS Index of Multiple Deprivation 2019 LSOA-level data and the LSOA-to-postcode lookup. Joins on postcode to add:
- `IMD_Decile` — 1 (most deprived) to 10 (least deprived)
- `IMD_Score` — raw deprivation score
- `Employment_Score` — employment domain deprivation
- `Income_Score` — income domain deprivation

In [ ]:
IMD_URL    = ("https://assets.publishing.service.gov.uk/government/uploads/system/uploads"
              "/attachment_data/file/833995/File_7_-_All_IoD2019_Scores__Ranks__Deciles_"
              "and_Population_Denominators_3.csv")
LSOA_URL   = ("https://geoportal.statistics.gov.uk/datasets/postcode-to-output-area-hierarchy"
              "-with-classifications-november-2011-lookup-in-england-and-wales"
              "/explore?location=52.900000,-2.000000,6.73")

print("Downloading IMD 2019 scores...")
imd_resp = requests.get(IMD_URL, timeout=60)
if imd_resp.status_code == 200:
    imd_df = pd.read_csv(io.StringIO(imd_resp.text))
    print(f"  IMD data loaded: {len(imd_df):,} LSOAs")
    print(f"  Columns: {imd_df.columns.tolist()[:8]} ...")
else:
    print(f"  ⚠ IMD download failed (HTTP {imd_resp.status_code}).")
    print("  To use IMD enrichment, manually download from:")
    print("  https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019")
    print("  and save as Data/imd_2019.csv")
    imd_df = None


In [ ]:
# LSOA to Postcode lookup (ONS Open Geography Portal)
LOOKUP_URL = ("https://opendata.arcgis.com/datasets/"
              "6a46e14a6c2441e3ab08c7b277335558_0.csv")

if imd_df is not None:
    print("Downloading LSOA-postcode lookup...")
    lookup_resp = requests.get(LOOKUP_URL, timeout=60)
    if lookup_resp.status_code == 200:
        lookup_df = pd.read_csv(io.StringIO(lookup_resp.text))
        print(f"  Lookup loaded: {len(lookup_df):,} postcodes")

        # Standardise postcode format (remove spaces, uppercase)
        lookup_df['pcds_clean'] = lookup_df['pcds'].str.replace(' ','').str.upper()
        df_combined['postcode_clean'] = (df_combined['RegAddress.PostCode']
                                         .astype(str).str.replace(' ','').str.upper())

        # Join postcode → LSOA
        df_combined = df_combined.merge(
            lookup_df[['pcds_clean','lsoa11cd']],
            left_on='postcode_clean', right_on='pcds_clean', how='left'
        )

        # Select IMD columns
        imd_cols = ['LSOA code (2011)',
                    'Index of Multiple Deprivation (IMD) Score',
                    'Index of Multiple Deprivation (IMD) Decile',
                    'Employment Score (rate)',
                    'Income Score (rate)']
        imd_cols_present = [c for c in imd_cols if c in imd_df.columns]
        imd_sub = imd_df[imd_cols_present].rename(columns={
            'LSOA code (2011)':                              'lsoa11cd',
            'Index of Multiple Deprivation (IMD) Score':    'IMD_Score',
            'Index of Multiple Deprivation (IMD) Decile':   'IMD_Decile',
            'Employment Score (rate)':                       'Employment_Score',
            'Income Score (rate)':                           'Income_Score',
        })

        # Join LSOA → IMD
        df_combined = df_combined.merge(imd_sub, on='lsoa11cd', how='left')
        df_combined = df_combined.drop(columns=['pcds_clean','postcode_clean','lsoa11cd'],
                                       errors='ignore')

        matched = df_combined['IMD_Decile'].notna().sum()
        print(f"  IMD matched: {matched:,} / {len(df_combined):,} "
              f"({matched/len(df_combined):.1%})")
        print(f"  Mean IMD Decile (liquidated): "
              f"{df_combined[df_combined.get('Target Value',df_combined.get('Target',None))==1]['IMD_Decile'].mean():.2f}")
        print(f"  Mean IMD Decile (active)    : "
              f"{df_combined[df_combined.get('Target Value',df_combined.get('Target',None))==0]['IMD_Decile'].mean():.2f}")
    else:
        print(f"  ⚠ Lookup download failed (HTTP {lookup_resp.status_code}).")
        print("  IMD columns will not be added — continuing without enrichment.")


## 4. Save Final Merged Dataset

In [ ]:
out_path = DATA_DIR / 'Merged_Final_Dataset.xlsx'
df_combined.to_excel(out_path, index=False)

print("=== Final Merged Dataset ===")
print(f"Rows    : {len(df_combined):,}")
print(f"Columns : {df_combined.shape[1]}")
print(f"\nNull counts (top 10):")
print(df_combined.isna().sum().sort_values(ascending=False).head(10).to_string())
print(f"\n✓ Saved to {out_path}")
